<a href="https://colab.research.google.com/github/lukwac123/machine_learning_bootcamp/blob/main/unsupervised/02_dimensionality_reduction/03_pca_wine.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px

np.set_printoptions(precision=4, suppress=True, edgeitems=5, linewidth=200)

In [2]:
df_raw = pd.read_csv('https://archive.ics.uci.edu/ml/machine-learning-databases/wine/wine.data', header=None)
df = df_raw.copy()
df.head()

,0,1,2,3,4,5,6,7,8,9,10,11,12,13
0,1,14.23,1.71,2.43,15.6,127,2.80,3.06,0.28,2.29,5.64,1.04,3.92,1065
1,1,13.20,1.78,2.14,11.2,100,2.65,2.76,0.26,1.28,4.38,1.05,3.40,1050
2,1,13.16,2.36,2.67,18.6,101,2.80,3.24,0.30,2.81,5.68,1.03,3.17,1185
3,1,14.37,1.95,2.50,16.8,113,3.85,3.49,0.24,2.18,7.80,0.86,3.45,1480
4,1,13.24,2.59,2.87,21.0,118,2.80,2.69,0.39,1.82,4.32,1.04,2.93,735


In [3]:
data = df.iloc[:, 1:]
target = df.iloc[:, 0]
data.head()

,1,2,3,4,5,6,7,8,9,10,11,12,13
0,14.23,1.71,2.43,15.6,127,2.80,3.06,0.28,2.29,5.64,1.04,3.92,1065
1,13.20,1.78,2.14,11.2,100,2.65,2.76,0.26,1.28,4.38,1.05,3.40,1050
2,13.16,2.36,2.67,18.6,101,2.80,3.24,0.30,2.81,5.68,1.03,3.17,1185
3,14.37,1.95,2.50,16.8,113,3.85,3.49,0.24,2.18,7.80,0.86,3.45,1480
4,13.24,2.59,2.87,21.0,118,2.80,2.69,0.39,1.82,4.32,1.04,2.93,735


In [4]:
target.value_counts()

,count
0,
2,71
1,59
3,48


In [5]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(data, target)

print(f'X_train shape: {X_train.shape}')
print(f'X_test shape: {X_test.shape}')

X_train shape: (133, 13)
X_test shape: (45, 13)


In [6]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_std = scaler.fit_transform(X_train)
X_test_std = scaler.transform(X_test)
X_train_std[:5]

array([[-0.7597, -1.0647, -0.6608, -0.1768, -0.864 ,  1.8961,  1.052 , -1.3414,  0.4462, -0.1577,  1.2373,  0.328 , -1.0303],
       [ 0.1065,  1.3126, -0.1183,  0.8718, -1.0059, -0.9977, -0.4501,  1.9668,  0.0011,  0.0082, -0.5461, -0.8996, -0.7129],
       [-0.5813, -0.5423, -1.2712,  0.2726, -1.0059, -0.1374, -0.1218, -0.3174, -0.2482, -0.8734,  0.3925,  1.3273, -0.1884],
       [-1.3965,  1.7044,  0.1191,  0.4224, -1.2187,  0.895 ,  0.9824, -1.1838,  2.2977, -0.9398, -0.9685,  1.4272, -1.164 ],
       [ 0.858 ,  0.6508,  0.6955, -1.3153,  1.1931,  0.6447,  0.9824, -1.4989,  0.0723,  0.1504,  0.0171,  1.0275,  0.3897]])

In [7]:
from sklearn.decomposition import PCA

pca = PCA(n_components=3)
X_train_pca = pca.fit_transform(X_train_std)
X_test_pca = pca.transform(X_test_std)
X_train_pca.shape

(133, 3)

In [8]:
results = pd.DataFrame(data={'explained_variance_ratio': pca.explained_variance_ratio_})
results['cumulative'] = results['explained_variance_ratio'].cumsum()
results['component'] = results.index + 1
results

,explained_variance_ratio,cumulative,component
0,0.350038,0.350038,1
1,0.198074,0.548112,2
2,0.113487,0.661599,3


In [9]:
fig = go.Figure(data=[go.Bar(x=results['component'], y=results['explained_variance_ratio'], name='explained variance ratio'),
                      go.Scatter(x=results['component'], y=results['cumulative'], name='cumulative explained variance')],
                layout=go.Layout(title=f'PCA - {pca.n_components_} components', width=950, template='plotly_dark'))
fig.show()

In [10]:
X_train_pca_df = pd.DataFrame(data=np.c_[X_train_pca, y_train], columns=['pca1', 'pca2', 'pca3', 'target'])
X_train_pca_df.head()

,pca1,pca2,pca3,target
0,2.034376,-1.842324,0.171873,2.0
1,-2.538566,0.067581,0.495454,2.0
2,0.381465,-2.009722,-0.280987,2.0
3,1.085329,-1.315491,1.688846,2.0
4,2.123020,1.208978,-0.343762,1.0


In [11]:
px.scatter_3d(X_train_pca_df, x='pca1', y='pca2', z='pca3', color='target', template='plotly_dark', width=950)

In [12]:
X_train_pca[:5]

array([[ 2.0344, -1.8423,  0.1719],
       [-2.5386,  0.0676,  0.4955],
       [ 0.3815, -2.0097, -0.281 ],
       [ 1.0853, -1.3155,  1.6888],
       [ 2.123 ,  1.209 , -0.3438]])

In [13]:
X_test_pca[:5]

array([[ 2.7491,  1.5842, -0.7472],
       [-3.1678,  0.9641, -0.0771],
       [-2.3974,  0.526 ,  1.1108],
       [ 4.3132,  2.1733, -1.5395],
       [ 2.2028,  2.0641,  0.005 ]])